In [39]:
# ── STEP 1 · Import Libraries ─────────────────
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# ── STEP 2 · Load Data ────────────────────────
data = fetch_openml(name="credit-g", version=1, as_frame=True, parser="auto")
df = data.frame.copy()

# ── STEP 3 · Prepare Features & Target ────────
X = pd.get_dummies(df.drop("class", axis=1))
y = (df["class"] == "bad").astype(int)

# Scale numerical columns
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# ── STEP 4 · Split Data ───────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── STEP 5 · Tune & Train Model ───────────────
param_grid = {
    "n_estimators":   [100, 200],
    "max_depth":      [3, 5],
    "learning_rate":  [0.05, 0.1],
}

grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("Best params:", grid.best_params_)

# ── STEP 6 · Evaluate ─────────────────────────
y_pred = best_model.predict(X_test)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["Good", "Bad"]))

Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}

Accuracy: 0.7750
              precision    recall  f1-score   support

        Good       0.81      0.89      0.85       140
         Bad       0.66      0.52      0.58        60

    accuracy                           0.78       200
   macro avg       0.74      0.70      0.71       200
weighted avg       0.77      0.78      0.77       200

